# Boston Public Alleys

Source: Boston Streetbooks Urban Planning Department Report, 2025.  
Geometry: OpenStreetMap via Overpass API (actual alley shapes, not approximations).  
Metadata: `../data/alleys.csv`.

---

In [1]:
# %pip install folium pandas requests  # uncomment if needed
import json
import re
from pathlib import Path

import folium
import pandas as pd
import requests

In [2]:
df = pd.read_csv("../data/alleys.csv", dtype={"zip": str})
# Build lookup by alley number for metadata enrichment
meta = df.set_index("number").to_dict(orient="index")
print(f"{len(df)} alleys in CSV.")

74 alleys in CSV.


In [3]:
OSM_CACHE = Path("../data/osm_alleys.json")

# Fetch both public and private alleys
OVERPASS_QUERY = '[out:json][timeout:60];way["name"~"(Public|Private) Alley"](42.30,-71.14,42.40,-70.98);out geom;'

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

HEADERS = {"User-Agent": "boston-alley-rankings/1.0 (personal research project)"}


def fetch_osm_alleys():
    if OSM_CACHE.exists():
        print(f"Loading cached OSM data from {OSM_CACHE}")
        with open(OSM_CACHE) as f:
            return json.load(f)

    for url in OVERPASS_ENDPOINTS:
        print(f"Trying {url} ...")
        try:
            resp = requests.post(url, data={"data": OVERPASS_QUERY},
                                 headers=HEADERS, timeout=90)
            if resp.status_code == 200:
                data = resp.json()["elements"]
                OSM_CACHE.parent.mkdir(parents=True, exist_ok=True)
                with open(OSM_CACHE, "w") as f:
                    json.dump(data, f, indent=2)
                print(f"Fetched {len(data)} ways from {url}. Saved to {OSM_CACHE}")
                return data
            print(f"  HTTP {resp.status_code}: {resp.text[:200]}")
        except requests.RequestException as e:
            print(f"  Error: {e}")

    raise RuntimeError("All Overpass endpoints failed.")


osm_ways = fetch_osm_alleys()
public  = [w for w in osm_ways if "Public"  in w["tags"].get("name", "")]
private = [w for w in osm_ways if "Private" in w["tags"].get("name", "")]
print(f"Public: {len(public)}  |  Private: {len(private)}")

Trying https://overpass-api.de/api/interpreter ...
Fetched 130 ways from https://overpass-api.de/api/interpreter. Saved to ../data/osm_alleys.json
Public: 86  |  Private: 44


In [4]:
# Parse alley number from OSM name tag — handles both "Public Alley No. 101" and "Public Alley 101"
_num_re = re.compile(r"Public Alley(?:\s+No\.?)?\s*(\d+)", re.IGNORECASE)


def parse_number(name):
    m = _num_re.search(name or "")
    return int(m.group(1)) if m else None


# Summarise what we got vs what we expected
osm_numbers = {parse_number(w["tags"].get("name")) for w in osm_ways}
osm_numbers.discard(None)
csv_numbers = set(df["number"])

print(f"OSM alleys found : {len(osm_numbers)}")
print(f"CSV alleys       : {len(csv_numbers)}")
print(f"In CSV, missing from OSM : {sorted(csv_numbers - osm_numbers)}")
print(f"In OSM, not in CSV       : {sorted(osm_numbers - csv_numbers)}")

OSM alleys found : 75
CSV alleys       : 74
In CSV, missing from OSM : [301, 401, 903]
In OSM, not in CSV       : [545, 717, 902, 904]


In [5]:
DISTRICT_COLORS = {
    "B.P.": "#2176AE",
    "Rox.": "#57B14B",
    "E.B.": "#E84855",
}


def infer_district(coords):
    """Rough geographic district from centroid of way coordinates."""
    if not coords:
        return None
    lat = sum(c[0] for c in coords) / len(coords)
    lon = sum(c[1] for c in coords) / len(coords)
    if lon > -71.020:
        return "E.B."   # East Boston, east of the harbor
    if lat < 42.338:
        return "Rox."   # Roxbury, south of Back Bay / South End
    return "B.P."


m = folium.Map(
    location=[42.3467, -71.0817],
    zoom_start=14,
    tiles="CartoDB positron",
)


def add_way(way, is_private):
    name   = way["tags"].get("name", "")
    number = parse_number(name)
    coords = [[n["lat"], n["lon"]] for n in way.get("geometry", [])]
    if not coords:
        return

    info     = meta.get(number, {})
    district = info.get("district") or infer_district(coords)
    color    = DISTRICT_COLORS.get(district, "#888888")

    popup_html = f"<b>{name}</b>"
    if info:
        popup_html += (
            f" ({info['district']})<br>"
            f"<i>From:</i> {info['from_desc']}<br>"
            f"<i>To:</i> {info['to_desc']}<br>"
            f"ZIP: {info['zip']} &nbsp; Ward: {info['ward']}"
        )
    else:
        popup_html += f"<br><i>District (inferred):</i> {district}"

    folium.PolyLine(
        coords,
        color=color,
        weight=4,
        opacity=0.85,
        dash_array="6 5" if is_private else None,
        tooltip=name,
        popup=folium.Popup(popup_html, max_width=340),
    ).add_to(m)


for way in osm_ways:
    is_private = "Private" in way["tags"].get("name", "")
    add_way(way, is_private)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
            background:white;padding:10px 14px;border-radius:6px;
            border:1px solid #ccc;font-size:13px;line-height:2;">
  <b>District</b><br>
  <span style='color:#2176AE'>&#9644;</span> B.P. (Boston Proper)<br>
  <span style='color:#57B14B'>&#9644;</span> Rox. (Roxbury)<br>
  <span style='color:#E84855'>&#9644;</span> E.B. (East Boston)<br>
  <br><b>Type</b><br>
  <span style='color:#555'>&#9644;</span> Solid = Public<br>
  <span style='color:#555'>- - -</span> Dashed = Private
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Public: {len(public)}  |  Private: {len(private)}")
m

Public: 86  |  Private: 44


In [6]:
# Summary table
(
    df[["number", "district", "zip", "ward", "from_desc", "to_desc"]]
    .sort_values("number")
    .reset_index(drop=True)
    .style.set_caption("Boston Public Alleys — Boston Streetbooks 2025")
)

,number,district,zip,ward,from_desc,to_desc
0,101,B.P.,02113,3,Cross Street between Commercial Street and Fulton Street,Richmond Street between Commercial Street and Fulton Street
1,102,B.P.,02108,3,Creek Square at the rear of 88 Blackstone Street,Marshall Street
2,301,B.P.,02114,5,92 Pinckney Street,70 River Street
3,303,B.P.,02114,5,46 Pinckney Street,approximately 65 ft SE / 72 ft NE / 74 ft SW (dead end)
4,401,B.P.,02116,4,8 Garrison Street,St. Botolph Street at Harcourt Street
5,402,B.P.,02116,4,13 Garrison Street,261 West Newton Street
6,403,B.P.,02116,4,10 Cumberland Street,258 West Newton Street
7,404,B.P.,02116,4,1 Cumberland Street,Public Alley No. 405
8,405,B.P.,02115,4,238 Huntington Avenue,203 St. Botolph Street
9,414,B.P.,02115,5,11 Hereford Street,opposite 27 Massachusetts Avenue
